# 4.6d The hard weeks: does weeks 5–15 of the pregnancy stand out?

§4.6.13 builds on §4.6.12 in [4.6c](roos_04.6c-graded-vis3-positive-test.ipynb): the symptom patterns (`SYMPTOM_PATTERNS`, `normalise`) and the Poisson-vs-negative-binomial fit (`fit_daily`) are explained there and imported here from `roos_046_shared.py`.

**Notebook 4.6 is split in four:** [a: move and Benji](roos_04.6a-move-and-baby.ipynb) (§4.6.1–4.6.9) · [b: symptoms and topics](roos_04.6b-symptoms-and-topics.ipynb) (§4.6.10–4.6.11) · [c: graded visualisation 3](roos_04.6c-graded-vis3-positive-test.ipynb) (§4.6.12) · [d: the hard weeks](roos_04.6d-hard-weeks.ipynb) (§4.6.13). Loading, the four dates, `save` and the helpers §4.6.13 reuses live in [roos_046_shared.py](roos_046_shared.py).

In [ ]:
import numpy as np
import pandas as pd
from goad_toolkit.analytics import NullDistribution, NullResult
from goad_toolkit.visualizer import (
    LinePlot,
    NullPlot,
    PlotSettings,
)
from roos_046_shared import (
    BABY_DATE,
    EVENT_DATE,
    MOVE_DATE,
    PREGNANCY_START,
    SIDE_COLORS,
    SYMPTOM_PATTERNS,
    fit_daily,
    load_own,
    local_chat,
    normalise,
    save,
)
from scipy import stats

from scripts.plots import BarPlot
from roos_pipeline.plots import EventMarker
from scripts.roos_plots import BeforeAfterDensityPlot, DotStripPlot


## Loading your data

Same chat as 4.6a: `load_own()` (in `roos_046_shared.py`) loads the export, merges Louis's three names into one, and adds `n_chars`.

In [ ]:
own = load_own()
chat = local_chat(own)
chat.head()

## 4.6.13 The hard weeks: does weeks 5–15 of the pregnancy stand out?

**My story: "How to prepare for a next pregnancy."** The difficult part of the pregnancy took
about ten weeks, from week 5 to week 15. This section checks whether the chat confirms that
memory: does that period stand out against the rest of our conversation, how strongly, and
which weeks were the worst?

**What the chat can and cannot tell.** It can show *when* the hard period started and ended,
*how intense* it was compared with normal times, and *which* symptoms dominated. It cannot
show what helped or what we should do differently; those lessons are mine to draw from it.

**Periods.** Pregnancy weeks are counted from `PREGNANCY_START` (§4.6.10: Benji's birth − 41
weeks = 28 July 2024), with the same convention `roos_03.3` used: week *n* starts at
`PREGNANCY_START` + *n* weeks.

| period | dates | why |
|---|---|---|
| before pregnancy (long-distance) | chat start – 28 Aug 2024 | the baseline: same channel, we lived apart |
| test to week 5 | 29 – 31 Aug 2024 | 3 days, left out of the comparison |
| **hard weeks 5–15 (long-distance)** | **1 Sep – 9 Nov 2024** | the period in question, 10 weeks |
| rest of pregnancy | 10 Nov 2024 – 10 May 2025 | from here on we lived together |
| after Benji | 11 May 2025 – end of the export | |

Choices flagged so I can overrule them:

- **"Until week 15"** is read as *up to the start of week 15*, which gives exactly the 10 weeks
  I remember.
- **The baseline includes weeks 0–4 of the pregnancy**, before we knew. If the symptoms
  already started then, that makes the baseline slightly higher, so the comparison below is
  conservative rather than flattering.
- **The main comparison is hard weeks vs. before**, because both are long-distance: anything
  we felt, we had to *write*. From the move onwards most of this was said in person, so the
  later periods are shown but are lower bounds.
- **Rates per 1,000 messages**, not only per day: after the move we wrote half as much, and a
  per-day rate would mostly measure that.

In [ ]:
# Pregnancy weeks counted from PREGNANCY_START (§4.6.10: BABY_DATE - 41 weeks), with the same
# convention roos_03.3 used: week n starts at PREGNANCY_START + n weeks.
HARD_START = PREGNANCY_START + pd.Timedelta(weeks=5)  # start of week 5
HARD_END = PREGNANCY_START + pd.Timedelta(weeks=15)  # end of week 14 = 10 weeks, exclusive

PERIODS = {  # label: [start, end)
    "before pregnancy (long-distance)": (chat["ts_local"].min().normalize(), EVENT_DATE),
    "test to week 5": (EVENT_DATE, HARD_START),
    "hard weeks 5-15 (long-distance)": (HARD_START, HARD_END),
    "rest of pregnancy": (HARD_END, BABY_DATE),
    "after Benji": (BABY_DATE, chat["ts_local"].max().normalize() + pd.Timedelta(days=1)),
}
PERIOD_ORDER = [p for p in PERIODS if p != "test to week 5"]  # the 3 days in between are left out
HARD = "hard weeks 5-15 (long-distance)"
BEFORE = "before pregnancy (long-distance)"


def period_of(ts: pd.Series) -> pd.Series:
    labels = pd.Series(pd.NA, index=ts.index, dtype="object")
    for name, (lo, hi) in PERIODS.items():
        labels[(ts >= lo) & (ts < hi)] = name
    return labels


# Every message in the chat, flagged with the symptom patterns from step 3 of §4.6.12.
whole = chat.copy()
whole["normalised"] = whole["message"].map(normalise)
for name, pattern in SYMPTOM_PATTERNS.items():
    whole[name] = whole["normalised"].str.contains(pattern)
whole["either"] = whole["nausea"] | whole["fatigue"]
whole["period"] = period_of(whole["ts_local"])

for name, (lo, hi) in PERIODS.items():
    days = (hi - lo).days
    n = int((whole["period"] == name).sum())
    print(f"{name:34s} {lo.date()} to {(hi - pd.Timedelta(days=1)).date()}   {days:4d} days   "
          f"{n:6,} messages   {n / days:5.1f}/day")

During the hard weeks the chat was also *busier* than in the long-distance year before
(96.6 against 83.9 messages a day). From the move onwards, volume drops to about 43 and then
34 a day: the in-person effect in one number, and the reason the comparisons below use a rate
per 1,000 messages.

In [ ]:
# One row per calendar day, zero days included, so "per day" means per calendar day.
days_all = (
    whole.groupby(whole["ts_local"].dt.normalize())[["nausea", "fatigue", "either"]].sum()
    .join(whole.groupby(whole["ts_local"].dt.normalize()).size().rename("messages"))
)
days_all = days_all.reindex(
    pd.date_range(days_all.index.min(), days_all.index.max(), freq="D"), fill_value=0
).rename_axis("day").reset_index()
days_all["period"] = period_of(days_all["day"])

totals = days_all.groupby("period")[["messages", "nausea", "fatigue", "either"]].sum().loc[PERIOD_ORDER]
n_days = days_all.groupby("period").size().loc[PERIOD_ORDER]
period_table = pd.DataFrame({
    "days": n_days,
    "messages/day": totals["messages"] / n_days,
    "nausea/day": totals["nausea"] / n_days,
    "fatigue/day": totals["fatigue"] / n_days,
    "either/day": totals["either"] / n_days,
    # per 1,000 messages: corrects for how much we wrote at all in that period
    "nausea per 1,000 msgs": 1000 * totals["nausea"] / totals["messages"],
    "fatigue per 1,000 msgs": 1000 * totals["fatigue"] / totals["messages"],
    "either per 1,000 msgs": 1000 * totals["either"] / totals["messages"],
    "symptom messages": totals["either"],
})
period_table.round(2)

**What this means.**

- **Nausea is almost exclusive to the hard weeks:** 5.2 mentions per 1,000 messages, against
  0.1–0.4 in every other period.
- **Fatigue doubled:** 17.9 per 1,000 against 8.2 before. Fatigue has a real baseline (ordinary
  tiredness counts too), so the doubling is the signal, not the level.
- **Either symptom:** 22.6 per 1,000 in the hard weeks. That is 2.7 times the long-distance
  baseline, and well above both later periods (7.3 and 10.0).
- **After Benji, fatigue (9.7 per 1,000) is above the before-pregnancy baseline**, even though
  most tiredness was by then talked about in person. That number is a lower bound.

In [ ]:
# Same shuffle test as §4.6.12, on the two long-distance periods only: the hard weeks against
# the whole long-distance time before the pregnancy.
pair = days_all[days_all["period"].isin([BEFORE, HARD])].copy()


def share_difference(frame: pd.DataFrame) -> float:
    """Symptom messages per 1,000 messages, hard weeks minus before."""
    sums = frame.groupby("period")[["either", "messages"]].sum()
    share = 1000 * sums["either"] / sums["messages"]
    return share[HARD] - share[BEFORE]


hard_null = NullDistribution(share_difference, n_iter=2000, seed=4).run(pair, label="period")
ratio = period_table.loc[HARD, "either per 1,000 msgs"] / period_table.loc[BEFORE, "either per 1,000 msgs"]
print(f"symptom messages per 1,000: hard weeks minus before = {hard_null.observed:+.1f} "
      f"(x{ratio:.1f}), two-sided p = {hard_null.p_value():.4f}")

A random split of the same days almost never produces a difference this large (p ≈ 0.001,
the smallest value 2,000 shuffles can resolve). As in §4.6.12, neighbouring days are not
fully independent, so read it as "very unlikely to be chance", not as an exact number.

### Week by week: would a hard week be unusual in normal times?

A period average can hide one extreme week. So each of the 10 hard weeks is judged on its
own, the lesson-4 way (§4.5.5, "a point in the tail is a question, not a verdict"):

1. **Fit a family to the normal weeks.** Every full week of the long-distance time before
   the test. Weekly symptom counts are counts, and how much we talk about being tired
   wanders from week to week, so the negative binomial is the candidate again (same
   gamma-Poisson mechanism as messages per day), checked against Poisson on AIC.
2. **Score each hard week by its tail probability under that family:** how often would a
   normal week produce *at least* this many symptom messages? This uses the family we
   committed to, not a z-score that silently assumes a normal distribution.

Weekly counts are compared directly because the hard weeks and the normal weeks have similar
volume (96.6 and 83.9 messages a day). The last column is a volume check anyway: the share
per 1,000 messages, as a percentile among the normal weeks.

In [ ]:
# One row per pregnancy-week-sized block over the whole chat (week 0 = PREGNANCY_START), so
# the hard period is exactly weeks 5-14 and every block is the same 7 weekdays.
days_all["week"] = (days_all["day"] - PREGNANCY_START).dt.days // 7
weekly = days_all.groupby("week").agg(
    start=("day", "min"), n_days=("day", "size"),
    messages=("messages", "sum"), either=("either", "sum"),
    nausea=("nausea", "sum"), fatigue=("fatigue", "sum"),
)
weekly = weekly[weekly["n_days"] == 7].copy()  # drop the partial first and last block
weekly["share"] = 1000 * weekly["either"] / weekly["messages"].where(weekly["messages"] > 0)
weekly["group"] = np.select(
    [weekly.index < 4, weekly.index.isin(range(5, 15))],
    ["before pregnancy", "hard weeks"], default="other",
)

baseline = weekly.loc[weekly["group"] == "before pregnancy", "either"].to_numpy().astype(float)
baseline_fit = fit_daily(baseline)  # same Poisson-vs-nbinom MLE as §4.6.12, on weekly counts
nb_week = baseline_fit["nb_dist"]
print(f"normal long-distance weeks: {len(baseline)} weeks, nbinom μ = {baseline_fit['nb_mu']:.1f} "
      f"symptom messages/week, r = {baseline_fit['nb_r']:.2f}  "
      f"(AIC nbinom {baseline_fit['nb_aic']:.0f} vs poisson {baseline_fit['poisson_aic']:.0f})")

hard_weeks = weekly[weekly["group"] == "hard weeks"].copy()
# Tail probability under the family fitted to normal weeks: how often would an ordinary
# long-distance week produce at least this many symptom messages? (§4.5.5: a point in the tail
# is judged against the family you committed to, not a z-score.)
hard_weeks["P(normal week ≥ this)"] = nb_week.sf(hard_weeks["either"] - 1)
hard_weeks["share percentile vs normal weeks"] = hard_weeks["share"].map(
    lambda s: (weekly.loc[weekly["group"] == "before pregnancy", "share"] < s).mean()
)
cutoff = nb_week.ppf(0.99)
print(f"99th percentile of a normal week: {cutoff:.0f} symptom messages; "
      f"hard weeks above it: {(hard_weeks['either'] > cutoff).sum()} of {len(hard_weeks)}")
hard_weeks["start"] = hard_weeks["start"].dt.date
hard_weeks.rename_axis("pregnancy week")[
    ["start", "messages", "either", "nausea", "fatigue", "P(normal week ≥ this)", "share percentile vs normal weeks"]
].round(4)

**What this means.**

- **The normal weeks fit a negative binomial** (μ = 4.9 symptom messages a week, r = 5.26),
  preferred over Poisson by ΔAIC ≈ 10.
- **6 of the 10 hard weeks are beyond the 99th percentile** of a normal week (more than 14
  symptom messages). The worst, **week 12 (from 20 October): 30 symptom messages**, which a
  normal week would essentially never produce.
- **The edges of the period are its mildest weeks:** week 5 (10) and week 14 (8). The chat
  confirms my memory of when it started and stopped.
- **Nausea came in waves:** none in weeks 9 and 11, peaks of 6–7 in weeks 6, 12 and 13.
  Fatigue was there every week (7 to 25 messages).
- **Volume doesn't explain it:** measured as a share of that week's messages, every hard week
  sits above at least 88% of the normal weeks.

In [ ]:
timeline_settings = PlotSettings(
    figsize=(12, 5),
    title="",
    xlabel="",
    ylabel="messages mentioning nausea or fatigue\nper 1,000 messages (weekly)",
)
host = LinePlot(timeline_settings)
fig, ax = host.plot(data=weekly, x="start", y="share", gaps=True, color="#7f7f7f", linewidth=1.2)
ax.axvspan(HARD_START, HARD_END, color=timeline_settings.highlight_color, alpha=0.15, lw=0)
hard_line = weekly[weekly["group"] == "hard weeks"]
ax.plot(hard_line["start"], hard_line["share"], color=timeline_settings.highlight_color, linewidth=2.5)
ax.text(HARD_START + (HARD_END - HARD_START) / 2, 0.97, "weeks 5-15", transform=ax.get_xaxis_transform(),
        ha="center", va="top", color=timeline_settings.highlight_color, fontweight="bold")
# MOVE_DATE is the notebook's approximate move date (§4.6.4), hence "around".
for date, label, height in [(str(MOVE_DATE.date()), "moved in together (around)", 0.68),
                            (str(BABY_DATE.date()), "Benji is born", 0.88)]:
    host.plot_on(EventMarker(timeline_settings), date=date, label=label, height=height, text_offset_days=20)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title(
    f"Weeks 5-15 stand out from the long-distance year before: {period_table.loc[HARD, 'either per 1,000 msgs']:.0f} "
    f"symptom messages per 1,000, against {period_table.loc[BEFORE, 'either per 1,000 msgs']:.0f}",
    loc="left", fontsize=12,
)
fig.text(0.01, -0.03, "After the move, tiredness was mostly talked about in person, so the chat undercounts it there; "
         "after Benji, weeks hold few messages, so one mention moves the line a lot.", fontsize=8.5, color="dimgray")
save(fig, "trimester_timeline")

The crimson stretch is the hard weeks; grey is everything else, the same grey-first rule as
§4.6.12.

- **Against the long-distance year before**, there is no comparable *sustained* run.
  Single weeks come close to the hard weeks' level, but never ten weeks in a row.
- **After Benji, single weeks spike higher than any hard week.** Those weeks hold few messages
  (about 34 a day), so one or two mentions move the line a lot, and the period as a whole
  stays at 10 per 1,000. That is noise from a small denominator, not a second hard period.
- **The "moved in together" marker is the notebook's approximate `MOVE_DATE`** (1 November
  2024, from §4.6.4). It falls inside the last weeks of the hard period, while I remember the
  move as mid-November. If the real date is later, the whole hard period was long-distance.

### Testing the stand-out chart before trusting it

Four checks, one per claim the next chart makes:

1. **Is the negative binomial a fair description of the normal weeks?** A parametric
   bootstrap. Draw 54 weeks from the fitted family, refit, and measure the largest gap between
   the data's cumulative curve and the fit's, 1,000 times. If the real data sit no further
   from their fit than these simulated samples do, the family is plausible.
2. **Does the 99% line hold in the real data**, not only in the fitted family?
3. **How surprising is "6 of 10"?** If the hard weeks were ordinary weeks, each would cross
   the line with the small probability from (2); the binomial gives the chance of 6 or more
   out of 10. This treats the weeks as independent, which neighbouring weeks are not quite,
   so read the result as an order of magnitude.
4. **Why the 10 hard weeks get no family of their own.** Fit one anyway, then resample the
   10 weeks 1,000 times and see how firmly the parameters are pinned down.

In [ ]:
rng = np.random.default_rng(4)
N_BOOT = 1000

# 1. Does the negative binomial actually describe the normal weeks? Parametric bootstrap:
#    draw 54 weeks from the fitted family, refit, and see how far the refitted family sits
#    from its own sample (largest gap between the two cumulative curves). If the real data
#    sit no further from their fit than these simulated samples do, the family is plausible.
def cdf_gap(sample: np.ndarray, dist) -> float:
    grid = np.arange(0, sample.max() + 1)
    empirical = np.searchsorted(np.sort(sample), grid, side="right") / len(sample)
    return float(np.abs(empirical - dist.cdf(grid)).max())


observed_gap = cdf_gap(baseline, nb_week)
boot_gaps = []
for _ in range(N_BOOT):
    sim = nb_week.rvs(len(baseline), random_state=rng).astype(float)
    boot_gaps.append(cdf_gap(sim, fit_daily(sim)["nb_dist"]))
gof_p = (np.array(boot_gaps) >= observed_gap).mean()
print(f"1. fit check, normal weeks: largest cdf gap {observed_gap:.3f}; "
      f"simulated samples from the family have a gap this large in {gof_p:.0%} of cases")

# 2. Does the 99% line hold in the real normal weeks, not only in the fitted family?
p_beyond = nb_week.sf(cutoff)  # P(normal week > cutoff) under the family
print(f"2. normal weeks above the 99% line ({cutoff:.0f}): {(baseline > cutoff).sum()} of {len(baseline)} "
      f"(the family expects {p_beyond * len(baseline):.1f})")

# 3. How likely are 6 or more out of 10 such weeks, if the hard weeks were normal weeks?
#    Treats the 10 weeks as independent, which neighbouring weeks are not quite.
n_above = int((hard_weeks["either"] > cutoff).sum())
p_six = stats.binom.sf(n_above - 1, len(hard_weeks), p_beyond)
print(f"3. chance that 10 normal weeks contain {n_above} or more beyond the line: {p_six:.1e}")

# 4. Why no family on the 10 hard weeks: fit one anyway, and resample the 10 weeks to see
#    how well its parameters are pinned down.
hard_counts = hard_weeks["either"].to_numpy().astype(float)
hard_fit = fit_daily(hard_counts)
boot_mu, boot_r = [], []
for _ in range(N_BOOT):
    resample = rng.choice(hard_counts, len(hard_counts), replace=True)
    f = fit_daily(resample)
    boot_mu.append(f["nb_mu"])
    boot_r.append(f["nb_r"])
lo_mu, hi_mu = np.percentile(boot_mu, [5, 95])
lo_r, hi_r = np.percentile(boot_r, [5, 95])
r_range = f"{lo_r:.1f}-{hi_r:.1f}" if hi_r < 1000 else f"{lo_r:.1f} to no upper limit (= Poisson)"
print(f"4. nbinom on the 10 hard weeks: μ = {hard_fit['nb_mu']:.1f} (90% range {lo_mu:.1f}-{hi_mu:.1f}), "
      f"r = {hard_fit['nb_r']:.1f} (90% range {r_range});  "
      f"AIC nbinom {hard_fit['nb_aic']:.1f} vs poisson {hard_fit['poisson_aic']:.1f}")

**What this means.**

1. **The family fits.** Simulated samples sit this far from their own fit in most cases, so
   the normal weeks are what a negative binomial produces.
2. **The line holds:** none of the 54 real normal weeks crosses it.
3. **6 of 10 by chance is practically impossible**, even allowing for the independence
   caveat. This is the test of the chart's claim.
4. **The 10 hard weeks are too few for a family:** a negative binomial and a Poisson fit
   them almost equally well (ΔAIC ≈ 1), and r can't be pinned down (anything from about 7 to
   no limit). What *is* pinned down is their average: about 15 symptom messages a week,
   against 4.9 in normal weeks. So the footnote under the chart is correct, and it says
   precisely what is too little: the data are enough for an average, not for a shape.

In [ ]:
standout_settings = PlotSettings(
    figsize=(10, 5.5),
    title="",
    xlabel="messages mentioning nausea or fatigue, per week",
    ylabel="share of normal weeks",
)
top = int(max(weekly.loc[weekly["group"] != "other", "either"].max(), baseline.max())) + 2
fig, ax = BeforeAfterDensityPlot(standout_settings).plot(
    samples={"before": baseline},
    bins=np.arange(-0.5, top + 1, 1),
    colors=SIDE_COLORS,
    densities={"before": nb_week.pmf},
    discrete=True,
    labels={"before": f"{len(baseline)} normal long-distance weeks, fitted negative binomial:\n"
                      f"average μ = {baseline_fit['nb_mu']:.1f} symptom messages a week\n"
                      f"r = {baseline_fit['nb_r']:.2f}: how much that average wanders\n"
                      f"from week to week (smaller r = more)"},
    label_positions={"before": (0.42, 0.97)},
)
ax.set_xlim(-0.5, top + 0.5)
ymax = ax.get_ylim()[1]
# One dot per hard week, stacked when two weeks share a count, so all 10 stay visible.
stack = hard_weeks.groupby("either").cumcount()
ax.scatter(hard_weeks["either"], ymax * (0.02 + 0.035 * stack), color=SIDE_COLORS["after"], s=60, zorder=3)
ax.axvline(cutoff, color="dimgray", linestyle=":", linewidth=1)
ax.text(cutoff, ymax * 0.35, " 99% of normal\n weeks below", color="dimgray", fontsize=9, va="center")
ax.text(0.42, 0.62, f"the 10 hard weeks (dots): {n_above} of 10\nbeyond what a normal week produces",
        transform=ax.transAxes, color=SIDE_COLORS["after"], fontweight="bold", va="top")
ax.set_title(f"{n_above} of the 10 hard weeks would be a 1-in-100 week in normal times",
             loc="left", fontsize=12, color="dimgray")
fig.suptitle("How to survive a pregnancy: prepare and plan.", x=0.125, ha="left",
             fontsize=15, fontweight="bold")
fig.text(0.01, -0.05,
         f"No family fitted to the 10 hard weeks: too few to tell a negative binomial from a Poisson. "
         f"Their average is clear: {hard_fit['nb_mu']:.1f} a week (90% range {lo_mu:.1f}-{hi_mu:.1f}).",
         fontsize=8.5, color="dimgray")
save(fig, "trimester_standout")

Grey bars: the 54 normal long-distance weeks, with the fitted negative binomial (μ = the average per week, r = how much that average wanders: a small r means very different weeks, a large r means a steady rate, and r → ∞ is a Poisson). Crimson dots:
the 10 hard weeks, stacked where two weeks share a count. Everything right of the dotted line
is something a normal week produces less than once in a hundred. This is the chart that
answers "does it stand out?" with the lesson-4 tools.

### Is the difference real? Shuffle the labels and see what chance does

The chart above compares the hard weeks with a *fitted family*. This check needs no family at
all. Put all 64 weeks (54 normal + 10 hard) in one pile, deal the "hard" label to 10 random
weeks, measure again, and repeat 10,000 times. If the labels mean nothing, the real result
should look like one of those random deals.

Three measurements, from loose to strict:

1. **The gap in averages:** hard weeks minus normal weeks, in symptom messages per week.
2. **The chart's own claim:** how many of the 10 "hard" weeks sit beyond the 99% line (14).
3. **Keeping the order of time:** shuffling treats weeks as interchangeable, but neighbouring
   weeks look alike (a tired week often follows a tired week). So the strictest version
   compares the hard weeks against *every run of 10 consecutive normal weeks*: did the year
   before ever produce a stretch like this on its own?

Plus two numbers about **overlap**: how often a random hard week beats a random normal week,
and how many hard weeks fall inside the normal weeks' range.

In [ ]:
# 64 weeks, one label each; NullDistribution shuffles the "group" column.
two_groups = weekly.loc[weekly["group"].isin(["before pregnancy", "hard weeks"]), ["either", "group"]]
N_SHUFFLE = 10_000


def mean_gap(frame: pd.DataFrame) -> float:
    """Symptom messages per week: hard-weeks average minus normal-weeks average."""
    means = frame.groupby("group")["either"].mean()
    return means["hard weeks"] - means["before pregnancy"]


def n_beyond_line(frame: pd.DataFrame) -> float:
    """How many weeks labelled 'hard' sit beyond the 99% line of the stand-out chart."""
    return float((frame.loc[frame["group"] == "hard weeks", "either"] > cutoff).sum())


gap_null = NullDistribution(mean_gap, n_iter=N_SHUFFLE, seed=7).run(two_groups, label="group")
beyond_null = NullDistribution(n_beyond_line, n_iter=N_SHUFFLE, seed=7).run(two_groups, label="group")

# The second test also has an exact answer: 6 of the 64 weeks are beyond the line, and a random
# deal of 10 labels is a hypergeometric draw. P(6 or more of those 6 land among the 10):
n_total_beyond = int((two_groups["either"] > cutoff).sum())
p_exact = stats.hypergeom.sf(n_above - 1, len(two_groups), n_total_beyond, len(hard_counts))

# 3. Order kept: every run of 10 consecutive normal weeks (they are consecutive, checked here).
assert (np.diff(weekly.index[weekly["group"] == "before pregnancy"]) == 1).all()
rolling_means = pd.Series(baseline).rolling(len(hard_counts)).mean().dropna().to_numpy()
window_result = NullResult(observed=hard_counts.mean(), null=rolling_means, n_iter=len(rolling_means))

# Overlap. Probability that a random hard week has more symptom messages than a random normal
# week (ties count half): 0.5 = the groups overlap completely, 1.0 = no overlap at all.
p_hard_higher = ((hard_counts[:, None] > baseline[None, :]).mean()
                 + 0.5 * (hard_counts[:, None] == baseline[None, :]).mean())
in_normal_range = int((hard_counts <= baseline.max()).sum())

# How precise is the gap? Resample both groups (bootstrap), 95% range of the gap in averages.
rng_gap = np.random.default_rng(7)
boot_gap = [rng_gap.choice(hard_counts, len(hard_counts)).mean() - rng_gap.choice(baseline, len(baseline)).mean()
            for _ in range(N_BOOT)]
lo_gap, hi_gap = np.percentile(boot_gap, [2.5, 97.5])

print(f"1. gap in averages: {hard_counts.mean():.1f} - {baseline.mean():.1f} = {gap_null.observed:+.1f} a week "
      f"(95% range {lo_gap:+.1f} to {hi_gap:+.1f}); shuffles this large: "
      f"{(gap_null.null >= gap_null.observed).sum()} of {N_SHUFFLE:,}, one-sided p <= {gap_null.p_value('greater'):.4f}")
print(f"2. hard weeks beyond the line: {n_above} of 10; shuffles with {n_above}+: "
      f"{(beyond_null.null >= n_above).sum()} of {N_SHUFFLE:,}; exact chance {p_exact:.1e}")
print(f"3. {len(rolling_means)} runs of 10 consecutive normal weeks: averages {rolling_means.min():.1f} to "
      f"{rolling_means.max():.1f}; hard weeks {hard_counts.mean():.1f}; runs at least as high: "
      f"{(rolling_means >= hard_counts.mean()).sum()} of {len(rolling_means)}")
print(f"overlap: a random hard week beats a random normal week in {p_hard_higher:.0%} of pairs; "
      f"{in_normal_range} of 10 hard weeks fall inside the normal range (0-{baseline.max():.0f})")

null_settings = PlotSettings(
    figsize=(15, 4.8),
    title="Is the difference real? What random labels produce (grey) against what happened (crimson)",
    subplot_titles=[
        "1. gap in averages, shuffled labels",
        "2. hard weeks beyond the 99% line, shuffled labels",
        "3. average of 10 consecutive normal weeks",
    ],
    subplot_xlabels=[
        "hard minus normal (symptom messages a week)",
        "number of 'hard' weeks beyond 14",
        "symptom messages a week",
    ],
    subplot_ylabels=["share of shuffles", "share of shuffles", "share of the 45 runs"],
    max_cols=3,
)
host = NullPlot(null_settings)
fig, axes = host.create_figure(n_plots=3)
panels = [
    (gap_null, dict(bins=30), "upper right",
     f"real gap {gap_null.observed:+.1f}\n{(gap_null.null >= gap_null.observed).sum()} of {N_SHUFFLE:,} shuffles reach it"),
    (beyond_null, dict(bins=np.arange(-0.5, 11.5, 1)), "upper center", f"real: {n_above} of 10\nexact chance {p_exact:.0e}"),
    (window_result, dict(bins=12), "upper center", f"hard weeks {hard_counts.mean():.1f}\nhighest normal run {rolling_means.max():.1f}"),
]
for ax, (result, hist_kwargs, legend_loc, label) in zip(axes, panels):
    host.plot_on_axes(NullPlot(null_settings), ax, result=result, label=label, stat="probability", **hist_kwargs)
    ax.legend(fontsize=8.5, loc=legend_loc)
    ax.spines[["top", "right"]].set_visible(False)
save(fig, "trimester_standout_shuffle")

**What this means.**

- **Shuffled labels never come close.** A random deal of the "hard" label gives a gap between
  −5 and +7 symptom messages a week. The real gap is **+10.4** (95% range +6.9 to +14.2), and
  none of the 10,000 shuffles reaches it, so p ≤ 0.0001. That is the smallest p this test can
  give, not an exact number.
- **The chart's claim holds without the fitted family.** Random labels put 0–2 "hard" weeks
  beyond the line, and almost never 4. Six is so rare that the shuffles never produce it; the
  exact chance is 2.8 · 10⁻⁶ (about 1 in 360,000). This is larger than the 3.4 · 10⁻¹¹ in
  check 3 because it uses only the 64 real weeks, not the negative binomial's thin tail. The
  conclusion is the same.
- **Even keeping the order of time, nothing comes close.** The 45 runs of 10 consecutive normal
  weeks average 3.7 to 7.5 symptom messages a week; the hard weeks average 15.3, twice the
  highest run. With only 45 runs the strongest statement available is "1 in 46" (p ≤ 0.02).
  This is the most honest test, because it does not pretend neighbouring weeks are independent.
- **The groups overlap, but little.** Pick a random hard week and a random normal week: the
  hard one has more symptom messages in **98%** of pairs. Yet 4 of the 10 hard weeks (8, 10,
  11 and 13 messages) fall inside the range of normal weeks. Those are the start, the end, and
  two quieter weeks in between. So "the period as a whole stands out" is solid, while "every
  hard week stands out" is not.
- **What p does *not* say:** it is not "the chance that something is going on". It is the
  chance of a gap this large *if the labels meant nothing*. Whether the cause is the pregnancy
  rather than, say, the end of the summer holidays is a question the shuffle cannot answer
  (see the confounders in 4.6c).

### Both groups as fitted families, and the shuffle result underneath

The stand-out chart again, reduced to its message: two fitted curves, one for the normal weeks
and one for the hard weeks, so the difference shows as two peaks far apart. No bars, so the
reader's eye goes to the gap rather than the week-to-week noise. Both groups get the same
family, a negative binomial, so the two curves differ only in their parameters.

With 10 weeks the hard-weeks curve is a sketch, not a finding: check 4 showed a Poisson fits
them just as well (ΔAIC ≈ 1), and r is barely pinned down. The *average* is what is solid, and
that is what the shuffle test at the bottom of the chart is about.

In [ ]:
both_settings = PlotSettings(
    figsize=(10, 6),
    title="",
    xlabel="messages mentioning nausea or fatigue, per week",
    ylabel="probability: share of weeks with exactly this many",
)
nb_hard = hard_fit["nb_dist"]
# Lines only: the fitted pmf of each group, evaluated at every whole number of messages.
curves = pd.DataFrame({"k": np.arange(0, top + 1)})
curves["normal"] = nb_week.pmf(curves["k"])
curves["hard"] = nb_hard.pmf(curves["k"])

host = LinePlot(both_settings)
fig, ax = host.plot(data=curves, x="k", y="normal", gaps=True, color=SIDE_COLORS["before"], linewidth=2.5)
host.plot_on(LinePlot(both_settings), data=curves, x="k", y="hard", gaps=True,
             color=SIDE_COLORS["after"], linewidth=2.5)
ax.set_xlim(-0.5, top + 0.5)
ax.set_ylim(0, None)
ax.spines[["top", "right"]].set_visible(False)

# Direct labels at each peak instead of a legend.
peak_normal, peak_hard = curves.loc[curves["normal"].idxmax()], curves.loc[curves["hard"].idxmax()]
ax.text(peak_normal["k"] + 1.5, peak_normal["normal"], "normal weeks", color=SIDE_COLORS["before"],
        fontweight="bold", va="center")
ax.text(peak_hard["k"] + 1.5, peak_hard["hard"], "hard weeks", color=SIDE_COLORS["after"],
        fontweight="bold", va="center")

# "three times" is written out, so check it still matches the data if anything upstream changes.
assert round(hard_counts.mean() / baseline.mean()) == 3
ax.set_title(f"In the hard weeks, nausea and fatigue came up three times as often: "
             f"{hard_counts.mean():.0f} messages a week instead of {baseline.mean():.0f}",
             loc="left", fontsize=12, color="dimgray")
fig.suptitle("How to survive a pregnancy: prepare and plan.", x=0.125, ha="left",
             fontsize=15, fontweight="bold")
fig.text(0.01, -0.02,
         f"Grey: {len(baseline)} normal long-distance weeks before the pregnancy (average {baseline.mean():.1f} a week). "
         f"Crimson: the {len(hard_counts)} hard weeks, pregnancy weeks 5-14 (average {hard_counts.mean():.1f}).\n"
         f"Both lines: fitted negative binomial.",
         fontsize=8.5, color="dimgray", va="top")
fig.text(0.01, -0.08,
         f"Chance alone does not explain this gap: when the 'hard' label was shuffled at random over all "
         f"{len(two_groups)} weeks (10,000 times),\na gap as large as the real {gap_null.observed:+.1f} symptom messages "
         f"a week occurred {(gap_null.null >= gap_null.observed).sum()} times (p ≤ {gap_null.p_value('greater'):.4f}).",
         fontsize=8.5, color="dimgray", va="top")
save(fig, "trimester_standout_both")

### The same weeks as two probability distributions

The chart above *is* a distribution: with `discrete=True` and one bin per count, each grey bar
is the share of the 54 normal weeks with exactly that many symptom messages, so the bars add up
to 1 and read as a probability, P(week = k). The fitted line is the negative binomial's pmf on
the same scale. Only the crimson dots are not: their height is just stacking, not a probability.

This version puts both groups on the probability scale: grey = the 54 normal weeks with their
fitted family, crimson = the 10 hard weeks (each week is 1/10 = 0.1, two weeks with the same
count make 0.2). No family on the crimson bars, for the reason in check 4 above.

In [ ]:
pmf_settings = PlotSettings(
    figsize=(10, 5.5),
    title="",
    xlabel="messages mentioning nausea or fatigue, per week",
    ylabel="probability: share of weeks with exactly this many",
)
fig, ax = BeforeAfterDensityPlot(pmf_settings).plot(
    samples={"normal": baseline, "hard": hard_counts},
    bins=np.arange(-0.5, top + 1, 1),
    colors={"normal": SIDE_COLORS["before"], "hard": SIDE_COLORS["after"]},
    densities={"normal": nb_week.pmf},
    discrete=True,
    labels={
        "normal": f"{len(baseline)} normal long-distance weeks\n"
                  f"line: fitted negative binomial (μ = {baseline_fit['nb_mu']:.1f}, r = {baseline_fit['nb_r']:.2f})",
        "hard": f"the {len(hard_counts)} hard weeks (no family fitted)\n"
                f"average {hard_counts.mean():.1f} a week",
    },
    label_positions={"normal": (0.58, 0.97), "hard": (0.58, 0.72)},
)
ax.set_xlim(-0.5, top + 0.5)
ax.axvline(cutoff, color="dimgray", linestyle=":", linewidth=1)
ax.text(cutoff, ax.get_ylim()[1] * 0.6, "99% of normal \nweeks below ", color="dimgray", fontsize=9,
        va="center", ha="right")
ax.set_title(f"Normal weeks peak at 2-8 symptom messages; {n_above} of the 10 hard weeks sit beyond {cutoff:.0f}",
             loc="left", fontsize=12, color="dimgray")
print(f"bars add up to: normal {np.histogram(baseline, bins=np.arange(-0.5, top + 1))[0].sum() / len(baseline):.2f}, "
      f"hard {np.histogram(hard_counts, bins=np.arange(-0.5, top + 1))[0].sum() / len(hard_counts):.2f}")
save(fig, "trimester_standout_pmf")

### Before, during and after: the ten weeks against both neighbours

The chart above compares the hard weeks with *normal times* only. This one adds the other
side: the weeks right after the hard period, until Benji's birth (pregnancy weeks 15–40; the
week he was born in is left out). Three columns:

- **Before the pregnancy** (long-distance, the same 54 weeks as above).
- **The hard weeks 5–15** (long-distance).
- **After them** (from about week 15 we lived together).

**Choices:**

- **One dot per week, nothing averaged away.** With only 10 hard weeks, a box or a single
  bar would hide how many of them actually stand out.
- **Per 1,000 messages:** after the move we wrote about half as much, so raw counts would
  mostly show that.
- **The black line is the period pooled** (all its symptom messages ÷ all its messages), the
  same rate as in the period table above.
- **Two shuffle tests**, hard weeks vs. before and hard weeks vs. after, at week level:
  shuffle which group a week belongs to, 2,000 times.

In [ ]:
# Three groups of full weeks: before (long-distance), the hard weeks, and the weeks right
# after them until Benji's birth (living together). Rate per 1,000 messages per week,
# because after the move we wrote about half as much.
baby_week = (BABY_DATE - PREGNANCY_START).days // 7  # the week Benji was born in, left out
weekly["compare"] = np.select(
    [weekly.index < 4, weekly.index.isin(range(5, 15)), weekly.index.isin(range(15, baby_week))],
    ["before", "hard weeks", "after"], default="",
)
compare = weekly[weekly["compare"] != ""].copy()
COMPARE_ORDER = ["before", "hard weeks", "after"]


def pooled_share(frame: pd.DataFrame) -> pd.Series:
    """Symptom messages per 1,000 messages, all weeks of a group pooled."""
    sums = frame.groupby("compare")[["either", "messages"]].sum()
    return 1000 * sums["either"] / sums["messages"]


rates = pooled_share(compare)
tests = {}
for other in ["before", "after"]:
    pair_weeks = compare[compare["compare"].isin(["hard weeks", other])]
    tests[other] = NullDistribution(
        lambda f, other=other: pooled_share(f).pipe(lambda s: s["hard weeks"] - s[other]),
        n_iter=2000, seed=4,
    ).run(pair_weeks, label="compare")
n_weeks = compare["compare"].value_counts()
for group in COMPARE_ORDER:
    print(f"{group:10s} {n_weeks[group]:3d} weeks   {rates[group]:5.1f} per 1,000 messages   "
          f"weeks above the hard-weeks' lowest week: {(compare.loc[compare['compare'] == group, 'share'] > compare.loc[compare['compare'] == 'hard weeks', 'share'].min()).sum()}")
for other, result in tests.items():
    print(f"hard weeks vs {other}: {result.observed:+.1f} per 1,000, shuffle p = {result.p_value():.4f}")

compare_settings = PlotSettings(
    figsize=(10, 5.5), title="", xlabel="",
    ylabel="messages mentioning nausea or fatigue\nper 1,000 messages (one dot = one week)",
    base_color="#7f7f7f",
)
fig, ax = DotStripPlot(compare_settings).plot(
    data=compare, x="compare", y="share", order=COMPARE_ORDER,
    highlight_groups=["hard weeks"],
    summaries=rates.to_dict(),
    summary_labels={g: f"{rates[g]:.1f}" for g in COMPARE_ORDER},
    tick_labels={
        "before": f"before the pregnancy\nlong-distance ({n_weeks['before']} weeks)",
        "hard weeks": f"weeks 5-15\nlong-distance ({n_weeks['hard weeks']} weeks)",
        "after": f"weeks 15-41, until Benji\ntogether ({n_weeks['after']} weeks)",
    },
)
ax.set_title(
    f"The ten hard weeks stand out on both sides: {rates['hard weeks']:.0f} symptom messages per 1,000, "
    f"against {rates['before']:.0f} before and {rates['after']:.0f} after",
    loc="left", fontsize=12,
)
fig.text(0.01, -0.06, "Black line = all weeks of that period pooled. From about week 15 we lived together, so tiredness "
         "was mostly said in person: the right column is a lower bound.", fontsize=8.5, color="dimgray")
save(fig, "trimester_before_during_after")

**What this means.**

- **The hard weeks stand out on both sides:** 22.6 symptom messages per 1,000, against 8.2
  before and 7.3 after.
- **Both differences are very unlikely to be chance** (p ≈ 0.001 each, the smallest value
  2,000 shuffles can resolve).
- **The overlap is small.** Only 6 of the 54 weeks before, and 1 of the 26 weeks after, come
  above the *mildest* hard week. The rest of the hard weeks sit above every other week of the
  pregnancy.

**One honest caveat about the right-hand column:** from about week 15 we lived together, and
most of the tiredness was then said in person, not written. So "after" is a lower bound, and
part of the drop could be the move rather than the symptoms easing. The **left-hand
comparison is the clean one** (same channel, both long-distance). The right-hand one shows
the chat never again reached that level during the pregnancy, but it can't say by how much
the symptoms themselves eased.

In [ ]:
plot_rows = period_table.reset_index().rename(columns={"period": "period"})
plot_rows["label"] = plot_rows["period"].str.replace(" (long-distance)", "\n(long-distance)", regex=False)
period_settings = PlotSettings(figsize=(10, 5), title="", xlabel="",
                               ylabel="symptom messages per 1,000 messages")
palette = {lab: (period_settings.highlight_color if per == HARD else "#7f7f7f")
           for lab, per in zip(plot_rows["label"], plot_rows["period"])}
fig, ax = BarPlot(period_settings).plot(
    data=plot_rows, x="label", y="either per 1,000 msgs", hue="label", palette=palette,
    saturation=1, legend=False,
)
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", padding=3)
ax.set_xlabel("")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("The hard weeks are the peak of the whole chat", loc="left", fontsize=12)
fig.text(0.01, -0.06, "Nausea or fatigue mentions per 1,000 messages. The last two periods are lower bounds: "
         "living together, most of this was said in person, not in the chat.", fontsize=8.5, color="dimgray")
save(fig, "trimester_periods")

### What this says for "how to prepare for a next pregnancy"

**What the data show**, with numbers from the cells above:

- **The hard period is real and has clear edges.** About 10 weeks of symptom talk at 2.7 times
  the normal rate. It started right at the positive test (week 5) and faded by week 14.
- **It peaked late, around week 12**, not at the start.
- **Nausea came and went; fatigue stayed.** Nausea had weeks with zero mentions; fatigue was
  there every single week.
- **We wrote more, not less**, during those weeks (96.6 against 83.9 messages a day), while we
  still lived apart.

**What the data cannot show:**

- **What helped**, or what would have made it easier.
- **How the later periods compare fairly.** We talked in person by then, so those bars are
  lower bounds.
- **Whether a next pregnancy will follow the same pattern.** This is n = 1 pregnancy.
- **Ordinary tiredness vs. pregnancy tiredness.** The fatigue keywords can't tell them apart;
  nausea is the cleaner signal.

**My lessons for a next pregnancy:** *(write them here, in my own words, from the points
above)*

### Reflectie-tekst

*(To be written once the fifth rubric criterion is confirmed from the course page: it
should refer by name to preattentive processing, text integration, storyline (verhaallijn),
spurious correlations, and that fifth criterion.)*